# NCAA Data Preparation

**Design Thinking Question:** Does 3-point shooting affect win rate in NCAA basketball?

This notebook prepares the raw NCAA data to enable analysis of the relationship between 3-point shooting and team success.

---

## Structure

**PREPARE**
1. Data Collection
2. Data Understanding
3. Data Modelling
4. Data Integration

**PROCESS**

5. Data Transformation
6. Data Cleaning
7. Feature Engineering

**Output:** `data/combined_ncaa.csv`

---
# PREPARE

## 1. Data Collection

**Source:** Kaggle — NCAA Basketball Dataset

Load the raw data sources used to answer the design thinking question.

| Source | Description |
|---|---|
| `KenPom Barttorvik.csv` | Per-year team efficiency ratings, 3PT stats, win/loss record, tournament seed and round reached (2008–2026) |
| `Shooting Splits.csv` | Shot-type breakdown by team — 3s vs 2s vs dunks, with FG% and share of total shots (2010–2026) |

In [ ]:
import pandas as pd
import numpy as np
import os

# Resolve data directory whether notebook is run from project root or ncaa/
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'ncaa', 'data')
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'data')

# Load raw data sources
kp = pd.read_csv(os.path.join(DATA_DIR, 'KenPom Barttorvik.csv'))
ss = pd.read_csv(os.path.join(DATA_DIR, 'Shooting Splits.csv'))

print('Data directory:', DATA_DIR)
print()
print(f'KenPom Barttorvik : {kp.shape[0]:,} rows  |  {kp.shape[1]} columns')
print(f'Shooting Splits   : {ss.shape[0]:,} rows  |  {ss.shape[1]} columns')
print()
print(f'KenPom years  : {sorted(kp["YEAR"].unique())}')
print(f'Splits years  : {sorted(ss["YEAR"].unique())}')

## 2. Data Understanding

Inspect the structure, data types, and contents of each raw source before selecting or transforming anything.

### KenPom Barttorvik — Column Dictionary

| Column | Type | Description |
|---|---|---|
| `YEAR` | int | Season year |
| `TEAM ID` | int | Unique team identifier |
| `TEAM` | str | Team name |
| `CONF` | str | Conference (e.g. ACC, SEC, B12) |
| `SEED` | float | NCAA tournament seed (1–16); NaN if team did not qualify |
| `ROUND` | int | Tournament round reached — encoded as bracket size: 0=No Tourney, 68=First Four, 64=R64, 32=R32, 16=S16, 8=E8, 4=F4, 2=Final, 1=Champion |
| `W` | int | Regular season + tournament wins |
| `L` | int | Regular season + tournament losses |
| `WIN%` | float | Win percentage (W / Games) |
| `GAMES` | int | Total games played |
| `KADJ EM` | float | **KenPom Adjusted Efficiency Margin** — net points per 100 possessions vs average D1 opponent (higher = better) |
| `KADJ O` | float | **KenPom Adjusted Offensive Efficiency** — points scored per 100 possessions, adjusted for opponent quality |
| `KADJ D` | float | **KenPom Adjusted Defensive Efficiency** — points allowed per 100 possessions, adjusted for opponent quality (lower = better) |
| `BARTHAG` | float | **Barttorvik win probability** vs an average D1 team (0–1 scale) |
| `BADJ EM` | float | **Barttorvik Adjusted Efficiency Margin** — similar to KADJ EM using Barttorvik's model |
| `3PT%` | float | Offensive 3-point field goal percentage |
| `3PTR` | float | **3PT attempt rate** — percentage of offensive FG attempts that are 3-pointers |
| `3PT%D` | float | Defensive 3PT FG% — percentage of opponent 3-pointers that go in |
| `3PTRD` | float | **Opponent 3PT attempt rate** — percentage of opponent shots that are 3-pointers |
| `EFG%` | float | **Effective FG% (offence)** — accounts for 3PT being worth more: (FGM + 0.5×3PM) / FGA |
| `EFG%D` | float | **Effective FG% (defence)** — opponent EFG% allowed |
| `2PT%` | float | Offensive 2-point field goal percentage |
| `2PT%D` | float | Defensive 2-point field goal percentage allowed |
| `2PTR` | float | 2PT attempt rate — share of shots that are 2-pointers |
| `2PTRD` | float | Opponent 2PT attempt rate |
| `KADJ T` | float | **KenPom Adjusted Tempo** — possessions per 40 minutes, adjusted for opponent pace |
| `K TEMPO` | float | Raw tempo — actual possessions per game |
| `PPPO` | float | **Points per possession (offence)** — raw scoring efficiency |
| `FT%` | float | Free throw percentage |
| `FTR` | float | **Free throw rate** — free throw attempts per field goal attempt |
| `TALENT` | float | Composite talent rating based on recruiting rankings |
| `EXP` | float | Team experience rating (weighted average years in program) |
| `WAB` | float | **Wins Above Bubble** — wins vs what a bubble-level team would earn on the same schedule |
| `ELITE SOS` | float | **Strength of schedule** — quality of opponents faced |

### Shooting Splits — Column Dictionary

| Column | Type | Description |
|---|---|---|
| `YEAR` | int | Season year |
| `TEAM ID` | int | Unique team identifier (joins to KenPom on YEAR + TEAM ID) |
| `THREES FG%` | float | Offensive 3-point FG% |
| `THREES SHARE` | float | Share of total offensive shot attempts that are 3-pointers (%) |
| `THREES FG%D` | float | Defensive 3PT FG% allowed |
| `THREES D SHARE` | float | Share of opponent shots that are 3-pointers (%) |
| `CLOSE TWOS FG%` | float | FG% on close 2-pointers (layups, short paint shots) — offence |
| `CLOSE TWOS SHARE` | float | Share of shots that are close 2-pointers (%) |
| `FARTHER TWOS FG%` | float | FG% on mid-range / farther 2-pointers — offence |
| `FARTHER TWOS SHARE` | float | Share of shots that are farther 2-pointers (%) |
| `DUNKS FG%` | float | FG% on dunk attempts — offence (typically near 100%) |
| `DUNKS SHARE` | float | Share of shots that are dunks (%) |

> **Note:** THREES SHARE + CLOSE TWOS SHARE + FARTHER TWOS SHARE + DUNKS SHARE ≈ 100% per team.
> Shooting Splits data starts from 2010; 2008–2009 rows will have NaN for all these columns.

In [ ]:
# Dataset coverage: teams per year, conference breakdown, tournament representation
print('=== Dataset Coverage ===')
print()
print('Teams per year:')
print(kp.groupby('YEAR').size().to_string())
print()
print(f'Total unique teams  : {kp["TEAM ID"].nunique()}')
print(f'Total conferences   : {kp["CONF"].nunique()}')
print(f'Year range          : {kp["YEAR"].min()} – {kp["YEAR"].max()}')
print()
print('Conference frequency (top 15):')
print(kp['CONF'].value_counts().head(15).to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of key KenPom columns
plot_cols = [
    ('WIN%',     'Win %'),
    ('KADJ EM',  'Adj. Efficiency Margin'),
    ('BARTHAG',  'Win Prob vs Avg D1'),
    ('3PT%',     '3PT FG%'),
    ('3PTR',     '3PT Attempt Rate'),
    ('3PT%D',    'Opp 3PT FG% Allowed'),
    ('3PTRD',    'Opp 3PT Attempt Rate'),
    ('EFG%',     'Eff. FG% (Off)'),
    ('EFG%D',    'Eff. FG% (Def)'),
    ('SEED',     'Tournament Seed'),
    ('K TEMPO',  'Tempo (possessions/game)'),
    ('WAB',      'Wins Above Bubble'),
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for ax, (col, label) in zip(axes, plot_cols):
    data = kp[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson', linewidth=1.5, linestyle='--', label=f'Mean {data.mean():.2f}')
    ax.axvline(data.median(), color='orange',  linewidth=1.5, linestyle=':',  label=f'Median {data.median():.2f}')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Distribution of Key Columns — KenPom Barttorvik', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Shooting Splits columns
ss_plot_cols = [
    ('THREES FG%',         '3PT FG% (Off)'),
    ('THREES SHARE',       '3PT Shot Share % (Off)'),
    ('THREES FG%D',        '3PT FG% Allowed (Def)'),
    ('THREES D SHARE',     'Opp 3PT Shot Share %'),
    ('CLOSE TWOS FG%',     'Close 2s FG%'),
    ('CLOSE TWOS SHARE',   'Close 2s Shot Share %'),
    ('FARTHER TWOS FG%',   'Farther 2s FG%'),
    ('FARTHER TWOS SHARE', 'Farther 2s Shot Share %'),
    ('DUNKS FG%',          'Dunks FG%'),
    ('DUNKS SHARE',        'Dunks Shot Share %'),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for ax, (col, label) in zip(axes, ss_plot_cols):
    data = ss[col].dropna()
    ax.hist(data, bins=40, color='mediumseagreen', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson', linewidth=1.5, linestyle='--', label=f'Mean {data.mean():.2f}')
    ax.axvline(data.median(), color='orange',  linewidth=1.5, linestyle=':',  label=f'Median {data.median():.2f}')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Distribution of Key Columns — Shooting Splits', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# KenPom Barttorvik â€” first look
print('=== KenPom Barttorvik ===')
print(f'Shape: {kp.shape}')
kp.head(3)

In [ ]:
# Column names, dtypes, null counts
kp.info()

In [ ]:
# Descriptive statistics for key 3PT and performance columns
kp_preview_cols = ['WIN%', 'KADJ EM', 'BARTHAG', '3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'EFG%D', 'SEED', 'ROUND']
kp[kp_preview_cols].describe().round(2)

In [ ]:
# Distribution of tournament round values
print('ROUND value distribution (0 = did not qualify):')
print(kp['ROUND'].value_counts().sort_index())

In [ ]:
# Shooting Splits â€” first look
print('=== Shooting Splits ===')
print(f'Shape: {ss.shape}')
ss.head(3)

In [ ]:
ss.info()

In [ ]:
# Shot type distributions
ss_preview_cols = ['THREES FG%', 'THREES SHARE', 'THREES FG%D', 'THREES D SHARE',
                   'CLOSE TWOS FG%', 'CLOSE TWOS SHARE', 'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
                   'DUNKS FG%', 'DUNKS SHARE']
ss[ss_preview_cols].describe().round(2)

### Single Column Profiling

Profile each relevant column individually — data type, missing rate, cardinality, and distribution statistics (numeric) or top values (categorical).

In [ ]:
def profile_numeric(df, col):
    """One-row summary for a numeric column."""
    s = df[col].dropna()
    return pd.Series({
        'dtype'   : str(df[col].dtype),
        'count'   : df[col].count(),
        'missing' : df[col].isna().sum(),
        'missing%': round(df[col].isna().mean() * 100, 2),
        'unique'  : df[col].nunique(),
        'min'     : round(s.min(),      3),
        'max'     : round(s.max(),      3),
        'mean'    : round(s.mean(),     3),
        'median'  : round(s.median(),   3),
        'std'     : round(s.std(),      3),
        'skew'    : round(s.skew(),     3),
        'kurtosis': round(s.kurtosis(), 3),
    }, name=col)

def profile_categorical(df, col):
    """One-row summary for a categorical / text column."""
    s = df[col]
    top = s.value_counts().head(5)
    return pd.Series({
        'dtype'    : str(s.dtype),
        'count'    : s.count(),
        'missing'  : s.isna().sum(),
        'missing%' : round(s.isna().mean() * 100, 2),
        'unique'   : s.nunique(),
        'top_1'    : f'{top.index[0]}  ({top.iloc[0]:,})' if len(top) > 0 else None,
        'top_2'    : f'{top.index[1]}  ({top.iloc[1]:,})' if len(top) > 1 else None,
        'top_3'    : f'{top.index[2]}  ({top.iloc[2]:,})' if len(top) > 2 else None,
        'top_4'    : f'{top.index[3]}  ({top.iloc[3]:,})' if len(top) > 3 else None,
        'top_5'    : f'{top.index[4]}  ({top.iloc[4]:,})' if len(top) > 4 else None,
    }, name=col)

print('profile_numeric() and profile_categorical() defined')

In [ ]:
# --- Single Column Profiling: KenPom Barttorvik ---

kp_num_cols = [
    'WIN%', 'W', 'L',
    'KADJ EM', 'KADJ O', 'KADJ D', 'BARTHAG', 'BADJ EM',
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'EFG%', 'EFG%D', '2PT%', '2PT%D', '2PTR', '2PTRD',
    'SEED', 'ROUND',
    'KADJ T', 'K TEMPO', 'PPPO', 'FT%', 'FTR',
    'TALENT', 'EXP', 'WAB', 'ELITE SOS', 'YEAR',
]
kp_cat_cols = ['CONF', 'TEAM']

print('── Numeric Columns ──')
kp_num_profile = pd.DataFrame([profile_numeric(kp, c) for c in kp_num_cols])
display(kp_num_profile)

print('\n── Categorical Columns ──')
kp_cat_profile = pd.DataFrame([profile_categorical(kp, c) for c in kp_cat_cols])
kp_cat_profile

In [ ]:
# --- Single Column Profiling: Shooting Splits ---
# All profiled columns here are numeric — no categorical columns in this source

ss_num_cols = [
    'THREES FG%', 'THREES SHARE', 'THREES FG%D', 'THREES D SHARE',
    'CLOSE TWOS FG%', 'CLOSE TWOS SHARE',
    'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
    'DUNKS FG%', 'DUNKS SHARE',
]

ss_num_profile = pd.DataFrame([profile_numeric(ss, c) for c in ss_num_cols])
ss_num_profile

### Multi-Column Profiling

Examine relationships between columns — correlation matrix across 3PT stats and performance metrics, and grouped mean profiles by tournament stage.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation matrix: 3PT stats vs performance metrics
corr_cols = [
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'EFG%', 'EFG%D', '2PT%', '2PT%D',
    'WIN%', 'KADJ EM', 'BARTHAG', 'KADJ O', 'KADJ D',
    'WAB', 'TALENT',
]

corr_matrix = kp[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix — 3PT Stats vs Performance Metrics', fontsize=13, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()
plt.show()

# Print strongest correlations with WIN%
print('
Top correlations with WIN%:')
win_corr = corr_matrix['WIN%'].drop('WIN%').sort_values(key=abs, ascending=False)
print(win_corr.to_string())

In [ ]:
# Grouped mean profile by tournament round
# Shows how average column values shift as teams go deeper in the tournament
round_labels = {
    0: 'No Tourney', 1: 'First Four', 2: 'R64',
    3: 'R32', 4: 'S16', 5: 'E8', 6: 'F4', 7: 'Final', 8: 'Champion'
}

group_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'WIN%', 'KADJ EM', 'BARTHAG']

kp_tmp = kp.copy()
kp_tmp['Stage'] = kp_tmp['ROUND'].map({0:0,68:1,64:2,32:3,16:4,8:5,4:6,2:7,1:8}).map(round_labels)
stage_order = list(round_labels.values())

grouped = (
    kp_tmp.groupby('Stage')[group_cols]
    .mean()
    .reindex(stage_order)
    .round(2)
)
print('Mean values by tournament stage:')
grouped

## 3. Data Modelling

Select only the columns relevant to answering the design thinking question. Reducing to relevant columns makes the integrated dataset easier to work with.

**From KenPom Barttorvik:** team identity, win rate, efficiency metrics, 3PT stats (offence + defence), pace, talent, tournament outcome.

**From Shooting Splits:** shot-type shares and FG% broken down by category (3s, close 2s, far 2s, dunks), both offence and defence.

In [ ]:
# KenPom Barttorvik â€” selected columns
kp_cols = [
    # Identity
    'YEAR', 'TEAM ID', 'TEAM', 'CONF',
    # Tournament outcome
    'SEED', 'ROUND',
    # Season record
    'W', 'L', 'WIN%', 'GAMES',
    # Efficiency
    'KADJ EM',   # KenPom adjusted efficiency margin
    'KADJ O',    # adjusted offensive efficiency
    'KADJ D',    # adjusted defensive efficiency
    'BARTHAG',   # win probability vs average D1 team
    'BADJ EM',   # Barttorvik adjusted efficiency margin
    # 3-point offence
    '3PT%',      # 3PT field goal %
    '3PTR',      # 3PT attempt rate (share of shots that are 3s)
    # 3-point defence
    '3PT%D',     # defensive 3PT FG% allowed
    '3PTRD',     # opponent 3PT attempt rate
    # Other shooting
    'EFG%',      # effective FG% offence
    'EFG%D',     # effective FG% defence
    '2PT%',  '2PT%D',
    '2PTR',  '2PTRD',
    # Pace
    'KADJ T',    # adjusted tempo
    'K TEMPO',   # raw tempo (for back-calculating shot volume)
    # Scoring rates (for volume derivation)
    'PPPO',      # points per possession offence
    'FT%',       # free throw %
    'FTR',       # free throw rate
    # Context
    'TALENT',
    'EXP',
    'WAB',       # wins above bubble
    'ELITE SOS', # strength of schedule
]

# Shooting Splits â€” selected columns
ss_cols = [
    'YEAR', 'TEAM ID',
    # 3s offence
    'THREES FG%',    'THREES SHARE',
    # 3s defence
    'THREES FG%D',   'THREES D SHARE',
    # 2s offence
    'CLOSE TWOS FG%',   'CLOSE TWOS SHARE',
    'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
    # Dunks
    'DUNKS FG%',     'DUNKS SHARE',
]

kp_sel = kp[kp_cols].copy()
ss_sel = ss[ss_cols].copy()

print(f'KenPom selected : {kp_sel.shape[0]:,} rows  x  {kp_sel.shape[1]} columns')
print(f'Splits selected : {ss_sel.shape[0]:,} rows  x  {ss_sel.shape[1]} columns')

## 4. Data Integration

Merge the two datasets on `YEAR` + `TEAM ID`.

- A **left join** on KenPom keeps all team-years even when Shooting Splits data is unavailable (2008â€“2009).
- After merging, flag rows where Shooting Splits data is missing.

In [ ]:
df = kp_sel.merge(ss_sel, on=['YEAR', 'TEAM ID'], how='left')

print(f'Merged shape     : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Years covered    : {df["YEAR"].min()} â€“ {df["YEAR"].max()}')
print(f'Teams per year   : {df.groupby("YEAR").size().mean():.1f} (avg)')

In [ ]:
# Verify merge: check nulls introduced from Shooting Splits
ss_only_cols = [c for c in ss_cols if c not in ['YEAR', 'TEAM ID']]
print('Missing values in Shooting Splits columns after merge:')
print(df[ss_only_cols].isna().sum())

---
# PROCESS

## 5. Data Transformation

Convert raw values into formats suitable for analysis.

- Map the `ROUND` column (encoded as bracket size) to an ordinal scale so higher = deeper tournament run.
- Create binary indicators for tournament participation and Sweet 16 advancement.
- Standardise the `WIN%` column name.

In [ ]:
# ROUND encoding in source data:
#   0  = did not qualify
#   68 = First Four (play-in)
#   64 = Round of 64  |  32 = Round of 32  |  16 = Sweet 16
#   8  = Elite 8      |   4 = Final Four   |   2 = Championship game
#   1  = Champion

round_map = {0: 0, 68: 1, 64: 2, 32: 3, 16: 4, 8: 5, 4: 6, 2: 7, 1: 8}
df['tournament_round'] = df['ROUND'].map(round_map)

# Binary: reached the main tournament bracket (64 teams or better)
df['made_tournament'] = (df['ROUND'] >= 64).astype(int)

# Binary: reached at least the Sweet 16
df['reached_s16'] = (df['tournament_round'] >= 4).astype(int)

# Standardise column name
df.rename(columns={'WIN%': 'win_pct'}, inplace=True)

print('tournament_round distribution:')
labels = {0:'No Tourney', 1:'First Four', 2:'R64', 3:'R32', 4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'}
print(df['tournament_round'].map(labels).value_counts().reindex(labels.values()))
print(f'\nTournament teams : {df["made_tournament"].sum():,} / {len(df):,}')
print(f'Sweet 16+ teams  : {df["reached_s16"].sum():,} / {len(df):,}')

## 6. Data Cleaning

Check data quality: duplicates, missing values, and coverage gaps. Flag rows where Shooting Splits data is unavailable so downstream analysis can filter appropriately.

In [ ]:
# Duplicate team-year rows
dupes = df.duplicated(subset=['YEAR', 'TEAM ID'])
print(f'Duplicate YEAR + TEAM ID rows: {dupes.sum()}')

In [ ]:
# Missing values across all columns
missing = df.isna().sum()
print('Columns with missing values:')
print(missing[missing > 0].to_string())

In [ ]:
# Flag rows that have Shooting Splits data
# (Shooting Splits only available from 2010 onward)
df['has_shot_splits'] = df['THREES FG%'].notna().astype(int)

coverage = df.groupby('YEAR')['has_shot_splits'].mean() * 100
print('Shooting Splits coverage by year (%):')
print(coverage.round(1).to_string())
print(f'\nRows WITH splits data    : {df["has_shot_splits"].sum():,}')
print(f'Rows WITHOUT splits data : {(df["has_shot_splits"] == 0).sum():,}')

## 7. Feature Engineering

Create derived columns that are more directly useful for analysing the 3-point shooting â†’ win rate relationship.

| Feature | Description |
|---|---|
| `three_rate_category` | Quartile bucket of 3PT attempt rate (Low â†’ High) |
| `three_pct_category` | Quartile bucket of 3PT FG% (Low â†’ High) |
| `three_pct_net` | Offensive 3PT% minus defensive 3PT% allowed |
| `three_rate_net` | Offensive 3PT attempt rate minus opponent's rate |
| `three_value` | Approximation of 3PT scoring contribution (efficiency Ã— volume) |
| `3PA` / `3PM` | Back-calculated season 3PT attempts and makes |
| `2PA` / `2PM` | Back-calculated season 2PT attempts and makes |

In [ ]:
# Quartile categories for 3PT volume and efficiency
df['three_rate_category'] = pd.qcut(
    df['3PTR'], q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

df['three_pct_category'] = pd.qcut(
    df['3PT%'], q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

print('3PT attempt rate quartiles:')
print(df['three_rate_category'].value_counts().sort_index())

In [ ]:
# Net differentials: how much better is your 3PT than your opponent's?
df['three_pct_net']  = df['3PT%']  - df['3PT%D']   # FG% advantage
df['three_rate_net'] = df['3PTR']  - df['3PTRD']   # attempt rate advantage

# 3PT value score: efficiency Ã— volume contribution
df['three_value'] = df['3PT%'] * 3 * (df['3PTR'] / 100)

print(df[['three_pct_net', 'three_rate_net', 'three_value']].describe().round(2))

In [ ]:
# Back-calculate season shot totals from efficiency rates
#
# Scoring identity:  Points = FGA Ã— (2 Ã— EFG% + FTR Ã— FT%)
#   â†’ FGA/game = PPPO Ã— K_TEMPO / (2 Ã— EFG% + FTR Ã— FT%)
#   â†’ FGA_season = FGA/game Ã— GAMES
#   â†’ 3PA = 3PTR Ã— FGA_season
#   â†’ 3PM = 3PT% Ã— 3PA

efg     = df['EFG%']   / 100
ftr     = df['FTR']    / 100
ft_pct  = df['FT%']    / 100
thr_r   = df['3PTR']   / 100
thr_pct = df['3PT%']   / 100
two_pct = df['2PT%']   / 100

fga_per_game = df['K TEMPO'] * df['PPPO'] / (2 * efg + ftr * ft_pct)
fga_season   = fga_per_game * df['GAMES']

df['3PA'] = (thr_r   * fga_season).round().astype('Int64')
df['3PM'] = (thr_pct * df['3PA']).round().astype('Int64')
df['2PA'] = (fga_season - df['3PA']).round().astype('Int64')
df['2PM'] = (two_pct * df['2PA']).round().astype('Int64')

print('Season shot volume averages:')
print(f'  3PA/game : {(df["3PA"] / df["GAMES"]).mean():.1f}')
print(f'  3PM/game : {(df["3PM"] / df["GAMES"]).mean():.1f}')
print(f'  2PA/game : {(df["2PA"] / df["GAMES"]).mean():.1f}')
print(f'  2PM/game : {(df["2PM"] / df["GAMES"]).mean():.1f}')

df[['TEAM', 'YEAR', 'GAMES', '3PTR', '3PT%', '3PA', '3PM', '2PA', '2PM']].head(8)

---
## Final Dataset Overview

In [ ]:
print('=== Combined NCAA Dataset ===')
print(f'Rows     : {df.shape[0]:,}')
print(f'Columns  : {df.shape[1]}')
print(f'Years    : {df["YEAR"].min()} â€“ {df["YEAR"].max()}')
print(f'Avg teams/year : {df.groupby("YEAR").size().mean():.1f}')
print()

print('3PT columns summary:')
three_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD',
              'three_pct_net', 'three_rate_net', 'three_value',
              'THREES FG%', 'THREES SHARE']
print(df[three_cols].describe().round(3))
print()

print('Success columns summary:')
success_cols = ['win_pct', 'KADJ EM', 'BARTHAG', 'SEED',
                'tournament_round', 'made_tournament', 'reached_s16']
print(df[success_cols].describe().round(3))

In [ ]:
df.head(5)

## Save Combined Dataset

In [ ]:
out_path = os.path.join(DATA_DIR, 'combined_ncaa.csv')
df.to_csv(out_path, index=False)

print(f'Saved  : {out_path}')
print(f'Shape  : {df.shape}')
print(f'\nAll columns:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}')